Task 2.0 - Linear Regreesion  
* Goal : Predict closing price for stock market using given features  
* Implementation : Created Linear Regression model using Gradient Descent
* Deadline - 20th Oct  
-Aparajaya

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print(f"Original train shape : {train.shape}")
print(f"Original test shape : {test.shape}")

* Step 1 : Clean the data by removing any rows with missing data (empty columns/features)

In [ ]:
features = ['high','low','momentum_index','beta_indicator','risk_premium','index','volatility_factor','technical_score','oscillator_value','liquidity_ratio','open','quant_index','trend_strength','market_sentiment','volume','alpha_signal']
target = 'close'

all_cols = features+[target]
train.dropna(subset=all_cols, inplace=True)
test.dropna(subset=all_cols, inplace=True)

#Removed features = 'date','symbols' - NaN

X_train = train[features].values
Y_train = train['close'].values
X_test = test[features].values
Y_test = test['close'].values

print(f"Cleaned X_train shape = {X_train.shape}")
print(f"Cleaned X_test shape = {X_test.shape}")

* Step 2 : Normalize the training and testing X and Y for better feature scaling

In [ ]:
mean_X = X_train.mean(axis=0)
mean_Y = Y_train.mean(axis=0)
sd_X = X_train.std(axis=0)
sd_Y = Y_train.std(axis=0)

l, = sd_X.shape
for i in range (l):
    if(sd_X[i].round() == 0):
        print(i)

sd_X[sd_X == 0] = 1
if sd_Y == 0:
    sd_Y = 1.0

X_train = (X_train - mean_X)/sd_X
Y_train = (Y_train - mean_Y)/sd_Y

In [ ]:
#Function to compute predicted Y based on inputted X
def predict(X,w,b):
    return np.dot(X,w)+b

In [ ]:
#Calculates Cost function
def cost(X,Y,Y_pred):
    m, = Y.shape
    J = (1/(2*m))*np.sum((Y_pred-Y)**2)
    return J

In [ ]:
#Calculates performance metrics like R^2, adjusted R^2, MSE and RMSE
def results(X,Y_pred,Y_actual):
    SS_res = np.sum((Y_actual - Y_pred)**2)
    SS_tot = np.sum((Y_actual - np.mean(Y_actual)) ** 2)
    m,n = X.shape

    r_squared = 1 - SS_res/SS_tot
    adj_r_squared = 1 - (1-r_squared) * (m-1) / (m-n-1)
    mse = ((Y_actual - Y_pred)**2).mean()
    rmse = math.sqrt(mse)

    return r_squared,adj_r_squared,mse,rmse

In [ ]:
#Computes Gradient Descent per epoch for given number of epochs
def gradient_descent(X,Y,w,b,alpha,epochs):
    m, = Y.shape
    J_history = []

    for i in range (epochs):
        Y_pred = predict(X,w,b)
        w_temp = w - (alpha/m)*np.dot(X.T,(Y_pred-Y))
        b_temp = b - (alpha/m)*np.sum(Y_pred-Y)
        w,b = w_temp,b_temp
        J = cost(X,Y,Y_pred)
        J_history.append(J) 

        if i%10 == 0 or i == epochs-1:
           
            r_squared,adj_r_squared,mse,rmse = results(X_train,Y_pred,Y_train)
            print(f"Epoch {i} : Cost={J:.6f} | R^2={r_squared:.4f} | Adjusted R^2={adj_r_squared:.4f} | Mean squared error={mse:.4f} | Root Mean squared error={rmse:.4f}")
    
    return w,b,J_history


* Step 3 : Initialize values of w,b,alpha and number of iterations   
* Step 4 : Call the gradient descent function (which further calls the predict and cost functions)  
* Step 5 : Print the final results for training data and store computed w,b

In [ ]:
initial_w = np.zeros(X_train.shape[1])
initial_b = 0
iterations = 100
alpha = 0.37

w_final,b_final,J_history = gradient_descent(X_train,Y_train,initial_w,initial_b,alpha,iterations)

print()
print("Final results for training :")
Y_pred = predict(X_train,w_final,b_final)
r_squared,adj_r_squared,mse,rmse = results(X_train,Y_pred,Y_train)
print(f"Cost={J_history[iterations-1]:.6f} | R^2={r_squared:.4f} | Adjusted R^2={adj_r_squared:.4f} | Mean squared error={mse:.4f} | Root Mean squared error={rmse:.4f}")

* Step 6 : Normalize the testing data and predict target (y) for testing data using computed w,b  
* Step 7 : Compute and print performance results for testing data

In [ ]:
X_test = (X_test - mean_X)/sd_X
Y_test = (Y_test - mean_Y)/sd_Y
Y_test_pred = predict(X_test,w_final,b_final)
r_squared,adj_r_squared,mse,rmse = results(X_test,Y_test_pred,Y_test)

print("Final results for testing :")
print(f"Cost={cost(X_test,Y_test,Y_test_pred):.6f} | R^2={r_squared:.4f} | Adjusted R^2={adj_r_squared:.4f} | Mean squared error={mse:.4f} | Root Mean squared error={rmse:.4f}")

Y_test_pred_actual = Y_test_pred * sd_Y + mean_Y

print()
print("Predicted Values :")
print(Y_test_pred_actual)

* Step 8 : Plot Cost vs Epochs graph  (shows if gradient descent is functioning properly)

In [ ]:
plt.plot(J_history)
plt.xlabel('Epochs')
plt.ylabel('Cost')
plt.title('Cost vs Epochs')
plt.show()

* Step 9 : Plot predicted vs actual values for both training and testing data  
         (If predicted values are close to red line - then predicted values are close to actual values)

In [ ]:
#Predicted vs Actual values for training data
sample_idx = np.random.choice(len(Y_train), size=2000, replace=False)

plt.figure(figsize=(6,6))
plt.scatter(Y_train[sample_idx], Y_pred[sample_idx], alpha=0.3, color='blue')
plt.plot([Y_train.min(), Y_train.max()], [Y_train.min(), Y_train.max()], 'r--', linewidth=2)
plt.xlabel("Actual Values (Y_train)")
plt.ylabel("Predicted Values (Y_pred)")
plt.title("Training Data: Predicted vs Actual")
plt.show()

#Predicted vs Actual values for testing data
sample_idx_test = np.random.choice(len(Y_test), size=2000, replace=False)

plt.figure(figsize=(6,6))
plt.scatter(Y_test[sample_idx_test], Y_test_pred[sample_idx_test], alpha=0.3, color='green')
plt.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], 'r--', linewidth=2)
plt.xlabel("Actual Values (Y_test)")
plt.ylabel("Predicted Values (Y_test_pred)")
plt.title("Testing Data: Predicted vs Actual")
plt.show()

* Step 10 : Plot Residuals vs Predicted values graph for testing data  
           (If residual = 0 - perfect prediction  
            If residual > 0 - underpredicted  
            If residual < 0 - overpredicted)

In [ ]:
residuals = Y_test - Y_test_pred
plt.scatter(Y_test_pred, residuals, alpha=0.7)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel("Predicted Close Price") 
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted Values")
plt.show()

In an ideal case, the residuals/errors should be completely random but they appear to be sloping upwards/have a funnel shape - its underpredicting the larger values - might be a non-linear function or an issue with feature scaling or choosing